In [ ]:
# 目的: CRITEO-UPLIFTv2を読み込み、行数・列・先頭行を確認する。
import pandas as pd

file_path = "../0.data/criteo-uplift-v2.1.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())
df.head()

In [3]:
# 目的: 利用可能な列名を再確認する。
print(df.columns.tolist())

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


In [4]:
# 目的: 特徴量と処置・outcomeの実データ例を目視する。
df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [5]:
# 目的: 型、処置・outcome構成、発生率、欠損を点検する。
print(df.dtypes)

print(df[["treatment", "conversion", "visit", "exposure"]].value_counts())

print(df[["treatment", "conversion", "visit", "exposure"]].mean())

print(df.isna().sum())

f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object
treatment  conversion  visit  exposure
1          0           0      0           11055129
0          0           0      0            2016832
1          0           1      0             385634
                       0      1             250702
                       1      1             154479
0          0           1      0              76042
1          1           1      1              23031
                              0              13680
0          1           1      0               4063
Name: count, dtype: int64
treatment     0.850000
conversion    0.002917
visit         0.046992
exposure      0.030631
dtype: float64
f0 

In [7]:
# 目的: baseline CausalForestDMLに必要なライブラリを読み込む。
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from econml.dml import CausalForestDML

In [8]:
# 旧探索コード: 100万train・300万testを抽出し、conversion用配列を作る。
# 注意: 下のv2 pipelineではvalidationを含む固定master splitに置き換える。
feature_cols = [f"f{i}" for i in range(12)]

sample_df = df.sample(
    n=500_000,
    random_state=42
).copy()

# 処置×アウトカムの構成比を保つ
sample_df["strata"] = (
    sample_df["treatment"].astype(str)
    + "_"
    + sample_df["conversion"].astype(str)
)

train_df = df.sample(
    n=1_000_000,
    random_state=42
).copy()

test_df = df.drop(index=train_df.index).sample(
    n=3_000_000,
    random_state=43
).copy()

X_train = train_df[feature_cols].to_numpy()
W_train = train_df["treatment"].to_numpy()
Y_train = train_df["conversion"].to_numpy()

X_test = test_df[feature_cols].to_numpy()
W_test = test_df["treatment"].to_numpy()
Y_test = test_df["conversion"].to_numpy()

In [9]:
# 旧baseline: conversionをoutcomeとしてCausalForestDMLを学習する。
cf = CausalForestDML(
    discrete_treatment=True,
    discrete_outcome=True,

    n_estimators=400,
    min_samples_leaf=100,
    max_depth=None,
    max_samples=0.45,

    honest=True,
    inference=True,

    cv=3,
    n_jobs=-1,
    random_state=42
)

cf.fit(
    Y_train,
    W_train,
    X=X_train
)

/opt/anaconda3/envs/thesis311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/thesis311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.ht

In [10]:
# 旧baseline: 独立test 300万件にconversion CATEを予測する。
cate_hat = cf.effect(X_test)

test_result = test_df[
    feature_cols + ["treatment", "conversion", "visit", "exposure"]
].copy()

test_result["cate_hat"] = cate_hat

test_result["cate_hat"].describe()

count    3.000000e+06
mean     1.005989e-03
std      4.749176e-03
min     -2.599943e-02
25%     -7.379819e-07
50%      7.131042e-06
75%      5.242193e-04
max      8.742833e-02
Name: cate_hat, dtype: float64

In [ ]:
# 旧baseline: conversion CATE順位の五分位別conversion ITTを確認する。
test_result["cate_group"] = pd.qcut(
    test_result["cate_hat"],
    q=5,
    labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
    duplicates="drop"
)

group_effect = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["conversion"]
    .agg(["mean", "count"])
    .reset_index()
)

group_means = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="mean"
)

group_counts = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="count"
)

group_means["observed_itt"] = (
    group_means[1] - group_means[0]
)

print(group_means)
print(group_counts)

treatment          0         1  observed_itt
cate_group                                  
Q1_low      0.000649  0.000807      0.000158
Q2          0.000044  0.000061      0.000017
Q3          0.000099  0.000159      0.000060
Q4          0.000297  0.000595      0.000299
Q5_high     0.008956  0.013380      0.004424
treatment       0       1
cate_group               
Q1_low      90843  509166
Q2          90328  509667
Q3          90748  509248
Q4          91022  508978
Q5_high     86865  513135


In [16]:
# 旧探索（参考のみ）: conversionモデルの順位でvisitを評価する。
# 注意: 正式なoutcome比較では使用せず、v2でvisitモデルを別途学習する。
test_result["cate_group"] = pd.qcut(
    test_result["cate_hat"],
    q=5,
    labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
    duplicates="drop"
)

group_effect = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["visit"]
    .agg(["mean", "count"])
    .reset_index()
)

group_means = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="mean"
)

group_counts = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="count"
)

group_means["observed_itt"] = (
    group_means[1] - group_means[0]
)

print(group_means)
print(group_counts)

treatment          0         1  observed_itt
cate_group                                  
Q1_low      0.030250  0.036248      0.005998
Q2          0.000775  0.000956      0.000181
Q3          0.003449  0.003766      0.000317
Q4          0.020061  0.023842      0.003781
Q5_high     0.142635  0.176188      0.033552
treatment       0       1
cate_group               
Q1_low      90843  509166
Q2          90328  509667
Q3          90748  509248
Q4          91022  508978
Q5_high     86865  513135


In [12]:
# 旧baseline: conversion五分位ごとのevent数とsample数を確認する。
group_summary = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["conversion"]
    .agg(
        conversion_rate="mean",
        conversions="sum",
        n="count"
    )
    .reset_index()
)

group_summary

,cate_group,treatment,conversion_rate,conversions,n
0,Q1_low,0,0.000649,59,90843
1,Q1_low,1,0.000807,411,509166
2,Q2,0,0.000044,4,90328
3,Q2,1,0.000061,31,509667
4,Q3,0,0.000099,9,90748
5,Q3,1,0.000159,81,509248
6,Q4,0,0.000297,27,91022
7,Q4,1,0.000595,303,508978
8,Q5_high,0,0.008956,778,86865
9,Q5_high,1,0.013380,6866,513135


In [13]:
# 目的: 全データの平均conversion ITTをOLS（HC1標準誤差）で推定する。
import statsmodels.formula.api as smf

model = smf.ols(
    "conversion ~ treatment",
    data=df
).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             conversion   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1123.
Date:                Sun, 09 Aug 2026   Prob (F-statistic):          3.27e-246
Time:                        18:53:56   Log-Likelihood:             2.0986e+07
No. Observations:            13979592   AIC:                        -4.197e+07
Df Residuals:                13979590   BIC:                        -4.197e+07
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0019   3.04e-05     63.804      0.0

In [14]:
# 目的: conversionのtreated/control平均差がOLS係数と一致するか確認する。
overall = df.groupby("treatment")["conversion"].agg(["mean", "count", "sum"])
print(overall)

itt = (
    df.loc[df["treatment"] == 1, "conversion"].mean()
    - df.loc[df["treatment"] == 0, "conversion"].mean()
)

print("ITT:", itt)

               mean     count    sum
treatment                           
0          0.001938   2096937   4063
1          0.003089  11882655  36711
ITT: 0.0011518730521316279


In [15]:
# 目的: 全データの平均visit ITTをOLS（HC1標準誤差）で推定する。
model_visit = smf.ols(
    "visit ~ treatment",
    data=df
).fit(cov_type="HC1")

print(model_visit.summary())

                            OLS Regression Results                            
Dep. Variable:                  visit   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     4996.
Date:                Sun, 09 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:56:20   Log-Likelihood:             1.8756e+06
No. Observations:            13979592   AIC:                        -3.751e+06
Df Residuals:                13979590   BIC:                        -3.751e+06
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0382      0.000    288.594      0.0

# Reproducible HTE pipeline v2

以下は上の探索的コードを残したまま追加した再現実験用pipelineです。`treatment`を処置とするITT/CITTを推定します。validationで仕様を選び、testは最終評価まで使用しません。conversionとvisitは必ず別モデルで学習します。

In [ ]:
# 目的: 再現実験のpath、特徴量、baseline設定、versionを一か所に固定する。
from pathlib import Path
from datetime import datetime, timezone
import json
import joblib
import platform
import time
import uuid

import econml
import numpy as np
import pandas as pd
import sklearn
from econml.dml import CausalForestDML

# Notebookから見た相対path。必要ならここだけ変更する。
DATA_PATH = Path("../0.data/criteo-uplift-v2.1.csv")
RESULTS_DIR = Path("../2.results/criteo_hte")
SPLIT_PATH = RESULTS_DIR / "master_split_seed42.npz"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [f"f{i}" for i in range(12)]
REQUIRED_COLS = FEATURE_COLS + ["treatment", "conversion", "visit", "exposure"]

BASELINE_CONFIG = {
    "learner": "CausalForestDML",
    "train_size": 300_000,  # smoke run。確認後に1M, 3Mへ変更
    "validation_size": 1_000_000,
    "test_size": 3_000_000,
    "split_seed": 42,
    "model_seed": 42,
    "n_estimators": 400,
    "min_samples_leaf": 100,
    "max_depth": None,
    "max_samples": 0.45,
    "cv": 3,
    "n_jobs": -1,
    "honest": True,
    "inference": True,
    "save_model": False,  # 感度分析では巨大modelを保存せず集計結果だけ残す
}

print({"python": platform.python_version(), "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "econml": econml.__version__})

In [ ]:
# 目的: schemaを検査し、全実験で共有するtrain/validation/test行を固定・保存する。
def validate_criteo_schema(data):
    missing = sorted(set(REQUIRED_COLS) - set(data.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if data[REQUIRED_COLS].isna().any().any():
        raise ValueError("Required columns contain missing values")
    for col in ["treatment", "conversion", "visit", "exposure"]:
        if not set(data[col].unique()) <= {0, 1}:
            raise ValueError(f"{col} is not binary")
    return True


def load_or_create_master_split(n_rows, path=SPLIT_PATH, seed=42,
                                validation_size=1_000_000, test_size=3_000_000):
    """一度作った行分割を再利用。train先頭n件によりsampleを入れ子にする。"""
    path = Path(path)
    if path.exists():
        saved = np.load(path)
        if int(saved["n_rows"]) != n_rows or int(saved["seed"]) != seed:
            raise ValueError("Saved split does not match dataset or seed")
        return saved["train_pool"], saved["validation"], saved["test"]
    if validation_size + test_size >= n_rows:
        raise ValueError("validation + test must be smaller than data")
    order = np.random.default_rng(seed).permutation(n_rows).astype(np.uint32)
    test_idx = order[:test_size]
    validation_idx = order[test_size:test_size + validation_size]
    train_pool_idx = order[test_size + validation_size:]
    np.savez_compressed(path, n_rows=n_rows, seed=seed, train_pool=train_pool_idx,
                        validation=validation_idx, test=test_idx)
    return train_pool_idx, validation_idx, test_idx


def prepare_arrays(data, row_idx, outcome):
    if outcome not in {"conversion", "visit"}:
        raise ValueError("outcome must be conversion or visit")
    part = data.iloc[row_idx]
    return (part[FEATURE_COLS].to_numpy(), part["treatment"].to_numpy(),
            part[outcome].to_numpy())


validate_criteo_schema(df)
train_pool_idx, validation_idx, test_idx = load_or_create_master_split(
    len(df), seed=BASELINE_CONFIG["split_seed"],
    validation_size=BASELINE_CONFIG["validation_size"],
    test_size=BASELINE_CONFIG["test_size"],
)
print({"train_pool": len(train_pool_idx), "validation": len(validation_idx),
       "test": len(test_idx), "split_path": str(SPLIT_PATH)})

In [ ]:
# 目的: Causal Forest学習、五分位評価、IPW uplift curve、AUUC/Qiniを共通関数化する。
def fit_causal_forest(data, train_indices, outcome, config):
    X_train, W_train, Y_train = prepare_arrays(data, train_indices, outcome)
    model = CausalForestDML(
        discrete_treatment=True, discrete_outcome=True,
        n_estimators=config["n_estimators"],
        min_samples_leaf=config["min_samples_leaf"],
        max_depth=config["max_depth"], max_samples=config["max_samples"],
        honest=config["honest"], inference=config["inference"],
        cv=config["cv"], n_jobs=config["n_jobs"],
        random_state=config["model_seed"],
    )
    started = time.perf_counter()
    model.fit(Y_train, W_train, X=X_train)
    return model, time.perf_counter() - started


def evaluate_quintiles(y, w, cate):
    y, w, cate = np.asarray(y).reshape(-1), np.asarray(w).reshape(-1), np.asarray(cate).reshape(-1)
    result = pd.DataFrame({"outcome": y, "treatment": w, "cate_hat": cate})
    # tieがあっても必ず同数の5群にする。outcomeはgroup作成に使わない。
    result["cate_group"] = pd.qcut(
        result["cate_hat"].rank(method="first"), 5,
        labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"]
    )
    rows = []
    for group_name, group in result.groupby("cate_group", observed=True):
        treated = group[group["treatment"] == 1]
        control = group[group["treatment"] == 0]
        rows.append({
            "cate_group": str(group_name), "n": len(group),
            "n_treated": len(treated), "n_control": len(control),
            "events": int(group["outcome"].sum()),
            "events_treated": int(treated["outcome"].sum()),
            "events_control": int(control["outcome"].sum()),
            "treated_rate": treated["outcome"].mean(),
            "control_rate": control["outcome"].mean(),
            "observed_uplift": treated["outcome"].mean() - control["outcome"].mean(),
            "mean_cate": group["cate_hat"].mean(),
        })
    return pd.DataFrame(rows)


def evaluate_uplift(y, w, cate, propensity=None):
    """IPW cumulative uplift。Qini = AUUC - random targeting line area。"""
    y, w, cate = np.asarray(y).reshape(-1), np.asarray(w).reshape(-1), np.asarray(cate).reshape(-1)
    propensity = float(w.mean()) if propensity is None else float(propensity)
    order = np.argsort(-cate, kind="stable")
    y_ordered, w_ordered = y[order], w[order]
    score = (w_ordered * y_ordered / propensity
             - (1 - w_ordered) * y_ordered / (1 - propensity))
    cumulative = np.r_[0.0, np.cumsum(score)]
    fraction = np.arange(len(cumulative)) / len(y)
    normalized = cumulative / len(y)
    auuc = float(np.trapezoid(normalized, fraction))
    qini = auuc - float(normalized[-1] / 2)
    curve = pd.DataFrame({"fraction_targeted": fraction,
                          "cumulative_uplift": cumulative})
    return {"auuc": auuc, "qini": qini}, curve

In [ ]:
# 目的: 1実験を学習・評価し、設定と結果を上書きせず保存する。
# 試行: conversionとvisitを別モデルで30万train・100万validation評価する。
def append_csv(frame, path):
    path = Path(path)
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def run_experiment(data, outcome, evaluation_indices, config, evaluation_split):
    experiment_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    train_indices = train_pool_idx[:config["train_size"]]
    model, fit_seconds = fit_causal_forest(data, train_indices, outcome, config)
    # 必要なrunだけ保存する。感度分析全件を保存すると数十GBになる。
    if config.get("save_model", False):
        joblib.dump(model, RESULTS_DIR / f"model_{experiment_id}.joblib", compress=3)
    X_eval, W_eval, Y_eval = prepare_arrays(data, evaluation_indices, outcome)
    started = time.perf_counter()
    cate = np.asarray(model.effect(X_eval)).reshape(-1)
    predict_seconds = time.perf_counter() - started
    quintiles = evaluate_quintiles(Y_eval, W_eval, cate)
    metrics, curve = evaluate_uplift(Y_eval, W_eval, cate)
    overall_itt = float(Y_eval[W_eval == 1].mean() - Y_eval[W_eval == 0].mean())
    summary = pd.DataFrame([{
        "experiment_id": experiment_id, "outcome": outcome,
        "evaluation_split": evaluation_split, "learner": config["learner"],
        "train_size": len(train_indices), "evaluation_size": len(evaluation_indices),
        "split_seed": config["split_seed"], "model_seed": config["model_seed"],
        "hyperparameters": json.dumps(config, sort_keys=True),
        "overall_itt": overall_itt, "mean_predicted_cate": float(cate.mean()),
        "auuc": metrics["auuc"], "qini": metrics["qini"],
        "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__, "econml_version": econml.__version__,
    }])
    quintiles.insert(0, "experiment_id", experiment_id)
    curve.insert(0, "experiment_id", experiment_id)
    append_csv(summary, RESULTS_DIR / "experiment_summary.csv")
    append_csv(quintiles, RESULTS_DIR / "quintile_results.csv")
    curve.to_parquet(RESULTS_DIR / f"uplift_curve_{experiment_id}.parquet", index=False)
    return model, cate, summary, quintiles, curve


# validationで仕様を選ぶ間はこちらを使う。test_idxは最終仕様確定まで渡さない。
# conversionとvisitをそれぞれ呼ぶため、必ず別モデルが学習される。
RUN_MODEL = False
if RUN_MODEL:
    validation_runs = {}
    for outcome_name in ["conversion", "visit"]:
        validation_runs[outcome_name] = run_experiment(
            df, outcome_name, validation_idx, BASELINE_CONFIG, "validation"
        )
else:
    print("RUN_MODEL=False: split/evaluation code only. Set True for the 300k smoke run.")

## Minimum robustness runs for the thesis/proposal

追加する最低ラインは、(1) conversionの200万train、(2) 同じ30万train行でmodel seed 123/2026、(3) 同じ30万train行でmin_samples_leaf 50/200、(4) 同じ30万train・validationでS/T/X-Learner比較。既存のseed 42・leaf 100結果は再利用し、重複runを避ける。すべてvalidation評価でありtestは使用しない。

In [ ]:
# 目的: 最低限のsample-size・seed・leaf感度分析条件を重複なく定義する。
MINIMUM_CF_RUNS = [
    {"label": "seed_123", "train_size": 300_000, "model_seed": 123, "min_samples_leaf": 100},
    {"label": "seed_2026", "train_size": 300_000, "model_seed": 2026, "min_samples_leaf": 100},
    {"label": "leaf_50", "train_size": 300_000, "model_seed": 42, "min_samples_leaf": 50},
    {"label": "leaf_200", "train_size": 300_000, "model_seed": 42, "min_samples_leaf": 200},
    {"label": "sample_2m", "train_size": 2_000_000, "model_seed": 42, "min_samples_leaf": 100},
]

def run_minimum_cf_robustness(data, runs=MINIMUM_CF_RUNS):
    completed = []
    for specification in runs:
        config = BASELINE_CONFIG.copy()
        config.update(specification)
        config["save_model"] = False
        print(f"START {specification['label']}: {config}", flush=True)
        result = run_experiment(data, "conversion", validation_idx, config, "validation")
        completed.append((specification["label"], result))
    return completed

RUN_MINIMUM_CF = False
if RUN_MINIMUM_CF:
    minimum_cf_results = run_minimum_cf_robustness(df)
else:
    print("RUN_MINIMUM_CF=False: set True to run 2M/seed/leaf robustness.")

In [ ]:
# 目的: 同じsplit・LightGBM設定でS/T/X-Learnerを比較可能にする。
from lightgbm import LGBMClassifier, LGBMRegressor

META_CONFIG = {
    "train_size": 300_000, "model_seed": 42,
    "n_estimators": 300, "learning_rate": 0.05,
    "num_leaves": 31, "min_child_samples": 100,
    "n_jobs": -1, "verbosity": -1,
}

def make_lgbm_classifier(config):
    return LGBMClassifier(n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"], num_leaves=config["num_leaves"],
        min_child_samples=config["min_child_samples"], n_jobs=config["n_jobs"],
        verbosity=config["verbosity"], random_state=config["model_seed"])

def make_lgbm_regressor(config):
    return LGBMRegressor(n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"], num_leaves=config["num_leaves"],
        min_child_samples=config["min_child_samples"], n_jobs=config["n_jobs"],
        verbosity=config["verbosity"], random_state=config["model_seed"])

def fit_predict_s_learner(X_train, W_train, Y_train, X_eval, config):
    model = make_lgbm_classifier(config)
    model.fit(np.column_stack([X_train, W_train]), Y_train)
    p1 = model.predict_proba(np.column_stack([X_eval, np.ones(len(X_eval))]))[:, 1]
    p0 = model.predict_proba(np.column_stack([X_eval, np.zeros(len(X_eval))]))[:, 1]
    return p1 - p0

def fit_predict_t_learner(X_train, W_train, Y_train, X_eval, config):
    model_1, model_0 = make_lgbm_classifier(config), make_lgbm_classifier(config)
    model_1.fit(X_train[W_train == 1], Y_train[W_train == 1])
    model_0.fit(X_train[W_train == 0], Y_train[W_train == 0])
    return model_1.predict_proba(X_eval)[:, 1] - model_0.predict_proba(X_eval)[:, 1]

def fit_predict_x_learner(X_train, W_train, Y_train, X_eval, config):
    # Kuenzelet al.型: arm別outcome model→imputed effect→arm別effect model。
    model_1, model_0 = make_lgbm_classifier(config), make_lgbm_classifier(config)
    treated, control = W_train == 1, W_train == 0
    model_1.fit(X_train[treated], Y_train[treated])
    model_0.fit(X_train[control], Y_train[control])
    d1 = Y_train[treated] - model_0.predict_proba(X_train[treated])[:, 1]
    d0 = model_1.predict_proba(X_train[control])[:, 1] - Y_train[control]
    tau_1, tau_0 = make_lgbm_regressor(config), make_lgbm_regressor(config)
    tau_1.fit(X_train[treated], d1)
    tau_0.fit(X_train[control], d0)
    propensity = float(W_train.mean())
    return propensity * tau_0.predict(X_eval) + (1 - propensity) * tau_1.predict(X_eval)


In [ ]:
# 目的: S/T/X-Learnerを同じ30万train・100万validationで実行し共通指標へ保存する。
def run_meta_learner_experiment(data, learner_name, outcome="conversion", config=META_CONFIG):
    experiment_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    train_indices = train_pool_idx[:config["train_size"]]
    X_train, W_train, Y_train = prepare_arrays(data, train_indices, outcome)
    X_eval, W_eval, Y_eval = prepare_arrays(data, validation_idx, outcome)
    fit_predict = {"S-Learner": fit_predict_s_learner,
                   "T-Learner": fit_predict_t_learner,
                   "X-Learner": fit_predict_x_learner}[learner_name]
    started = time.perf_counter()
    cate = np.asarray(fit_predict(X_train, W_train, Y_train, X_eval, config)).reshape(-1)
    runtime_seconds = time.perf_counter() - started
    quintiles = evaluate_quintiles(Y_eval, W_eval, cate)
    metrics, curve = evaluate_uplift(Y_eval, W_eval, cate)
    summary = pd.DataFrame([{
        "experiment_id": experiment_id, "outcome": outcome,
        "evaluation_split": "validation", "learner": learner_name,
        "train_size": len(train_indices), "evaluation_size": len(validation_idx),
        "split_seed": BASELINE_CONFIG["split_seed"], "model_seed": config["model_seed"],
        "hyperparameters": json.dumps(config, sort_keys=True),
        "overall_itt": float(Y_eval[W_eval == 1].mean() - Y_eval[W_eval == 0].mean()),
        "mean_predicted_cate": float(cate.mean()),
        "auuc": metrics["auuc"], "qini": metrics["qini"],
        "fit_seconds": runtime_seconds, "predict_seconds": 0.0,
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__, "econml_version": econml.__version__,
    }])
    quintiles.insert(0, "experiment_id", experiment_id)
    curve.insert(0, "experiment_id", experiment_id)
    append_csv(summary, RESULTS_DIR / "experiment_summary.csv")
    append_csv(quintiles, RESULTS_DIR / "quintile_results.csv")
    curve.to_parquet(RESULTS_DIR / f"uplift_curve_{experiment_id}.parquet", index=False)
    print(summary[["learner", "qini", "auuc", "fit_seconds"]].to_string(index=False))
    return cate, summary, quintiles, curve

RUN_META_LEARNERS = False
if RUN_META_LEARNERS:
    meta_results = {name: run_meta_learner_experiment(df, name)
                    for name in ["S-Learner", "T-Learner", "X-Learner"]}
else:
    print("RUN_META_LEARNERS=False: set True for S/T/X comparison.")

## Latest validated runs (2026-08-20)

同一validation 100万件で、train 30万・100万のconversion/visitを別モデルとして実行済み。30万→100万でconversion Qiniは0.000292→0.000511、Q5 observed upliftは0.004114→0.005213。visit Qiniは0.004125→0.004275、Q5 observed upliftは0.036588→0.036886。test 300万件は未使用。nuisance LogisticRegressionの収束警告が両サイズ・両outcomeで発生したため、正式な300万run前に対策条件を比較する。

In [ ]:
# 目的: 保存済みexperimentを読み、sample size・outcome間の結果を比較する。
experiment_summary = pd.read_csv(RESULTS_DIR / "experiment_summary.csv")
quintile_results = pd.read_csv(RESULTS_DIR / "quintile_results.csv")
display(experiment_summary[["experiment_id", "outcome", "train_size",
                            "overall_itt", "mean_predicted_cate",
                            "auuc", "qini", "fit_seconds"]])
display(quintile_results[["experiment_id", "cate_group", "events",
                          "observed_uplift", "mean_cate"]])

## Final test rule

validation結果を見てlearner・hyperparameter・thresholdを確定するまでは `test_idx` を評価に使わない。最終仕様確定後のみ `run_experiment(df, outcome, test_idx, final_config, "test")` を実行する。test結果を見て仕様を変更した場合、そのtestは以後validation扱いとなる。